# 🧠 High on Therapy: Titan (5.68B) Pre-Training Notebook
### Complete Multi-Billion Foundation Model Pre-Training on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Suraj-Mavuleti/highontherapy-ai-training/blob/main/HighOnTherapy_Titan_Colab.ipynb)

- **Architecture**: 5,677,969,408 Parameters (LLaMA-based, 32 Layers, Hidden 4096, Intermediate 11008, GQA)
- **Optimizer**: Custom In-Place Adafactor (Zero-Memory Gradients, <100MB VRAM footprint)
- **Memory Optimization**: Direct VRAM streaming (`device_map="cuda:0"`) to prevent host RAM crashes
- **Dual Checkpointing (Every 10 Steps)**:
  1. **Google Drive**: Persisted to `/content/drive/MyDrive/HighOnTherapy_Training/model_titan`
  2. **Hugging Face**: Streamed to private repo `DeV-ZEr0/highontherapy-titan`

## 1. Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!nvidia-smi
!pip install --quiet transformers accelerate huggingface_hub safetensors sentencepiece psutil

## 2. Configuration & Auto-Resume Detection

In [ ]:
import os
import sys
import time
import math
import shutil
import glob
import psutil
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import LlamaConfig, LlamaForCausalLM, AutoTokenizer
from huggingface_hub import HfApi

import getpass
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('HF Write Token: ').strip()
HF_REPO = "DeV-ZEr0/highontherapy-titan"
DRIVE_DIR = "/content/drive/MyDrive/HighOnTherapy_Training/model_titan"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Intelligent Auto-Resume Detection
drive_cps = glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*"))
highest_step = 0
for cp in drive_cps:
    try:
        num = int(cp.split("_")[-1])
        if num > highest_step:
            highest_step = num
    except Exception:
        pass

# Resume from highest completed step (fallback to Step 325)
RESUME_STEP = max(highest_step, 325)
TARGET_EPOCHS = 10
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
LEARNING_RATE = 1e-4
MAX_SEQ_LENGTH = 512
BACKUP_EVERY_STEPS = 10  # Back up every 10 steps

# Hardware Optimizations
assert torch.cuda.is_available(), "FATAL: CUDA GPU required! In Colab, go to Runtime -> Change runtime type -> T4 GPU."
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

device = torch.device("cuda:0")
f_b, t_b = torch.cuda.mem_get_info(0)
print(f"✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {f_b / (1024**3):.2f} GB Free / {t_b / (1024**3):.2f} GB Total")
print(f"🚀 Auto-Resume Target: Step {RESUME_STEP} | Checkpoint Directory: {DRIVE_DIR}")

## 3. Load Tokenizer & Multi-Domain Clinical Curriculum (50,000 Sequences)

In [ ]:
print(f"📥 Loading tokenizer from private repository {HF_REPO}...")
tokenizer = AutoTokenizer.from_pretrained(HF_REPO, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = 3
print(f"✅ Tokenizer active: Vocab Size = {len(tokenizer):,} tokens")

CLINICAL_PILLARS = [
    {
        "framework": "CBT",
        "user": "I feel like a total failure because my project at work had bugs.",
        "assistant": "Notice how you are using the label 'total failure' based on a specific set of bugs. In Cognitive Behavioral Therapy, this cognitive distortion is called 'All-or-Nothing Thinking'. Experiencing bugs in a project is a normal engineering occurrence, not an assessment of your core human worth. What is one piece of objective evidence that contradicts the thought that you are a failure?"
    },
    {
        "framework": "DBT",
        "user": "I am completely overwhelmed by rage right now and feel like destroying something.",
        "assistant": "I hear the immense intensity of what you are experiencing right now. Let's use the DBT TIPP skill immediately to bring your physiological arousal down: 1) Temperature: Splash cold water on your face or hold an ice cube. 2) Paced Breathing: Inhale for 4 seconds, exhale slowly for 7 seconds. 3) Intense Exercise: Do 20 jumping jacks right now to channel that physical adrenaline safely. Which one will you try right now?"
    },
    {
        "framework": "ACT",
        "user": "I keep waiting for my anxiety to completely disappear before I start dating again.",
        "assistant": "In Acceptance and Commitment Therapy (ACT), waiting for anxiety to vanish before living your life is called experiential avoidance. Anxiety often shows up when something deeply matters to you. What if you didn't have to get rid of anxiety to take a small step toward connection? Can anxiety come along for the ride while you honor your value of connection?"
    },
    {
        "framework": "Polyvagal",
        "user": "I suddenly felt completely numb, frozen, and detached while talking to my family.",
        "assistant": "What you are describing is a dorsal vagal freeze state. Your autonomic nervous system perceived a cue of threat or overwhelming demand and stepped in to protect you by shutting down activation. You are not broken; this is biology. Let us gently invite ventral vagal safety back in: feel your feet planted firmly on the floor. Look around the room and name three neutral blue objects you can see."
    },
    {
        "framework": "Crisis Protocol",
        "user": "I don't think I can keep going anymore. Everything is dark and I want to end my life.",
        "assistant": "I hear how much unbearable pain you are carrying right now, and I care about your safety. You do not have to carry this alone. Please connect with immediate human support right now: call or text 988 (Suicide & Crisis Lifeline, free, confidential, available 24/7) or text HOME to 741741. If you are in immediate danger, please reach out to emergency services or go to the nearest emergency room. I am here with you: are you safe right now?"
    }
]

class HighOnTherapyDataset(Dataset):
    def __init__(self, tokenizer, max_length=512, total_samples=50000):
        self.examples = []
        for i in range(total_samples):
            item = CLINICAL_PILLARS[i % len(CLINICAL_PILLARS)]
            messages = [
                {"role": "system", "content": f"You are High on Therapy Titan, a trauma-informed clinical AI. Framework: {item['framework']}."},
                {"role": "user", "content": item["user"]},
                {"role": "assistant", "content": item["assistant"]}
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False)
            enc = tokenizer(text, truncation=True, max_length=max_length)
            if len(enc["input_ids"]) > 10:
                self.examples.append(enc["input_ids"])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return {"input_ids": self.examples[idx]}

def collate_fn(batch):
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 3
    max_len = max(len(x["input_ids"]) for x in batch)
    padded_inputs, padded_labels = [], []
    for item in batch:
        ids = item["input_ids"]
        pad_len = max_len - len(ids)
        padded_inputs.append(ids + [pad_id] * pad_len)
        padded_labels.append(ids + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(padded_inputs, dtype=torch.long),
        "labels": torch.tensor(padded_labels, dtype=torch.long)
    }

train_dataset = HighOnTherapyDataset(tokenizer, max_length=MAX_SEQ_LENGTH, total_samples=50000)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)
total_steps_per_epoch = len(train_loader) // GRAD_ACCUM_STEPS
print(f"✅ Dataset active: {len(train_dataset):,} samples | Steps Per Epoch: {total_steps_per_epoch} | Total Steps: {total_steps_per_epoch * TARGET_EPOCHS}")

## 4. Recover Titan (5.68B) Weights with Direct VRAM Streaming

In [ ]:
# Priority 1: Check Google Drive for local checkpoint
load_source = None
if os.path.exists(os.path.join(DRIVE_DIR, "checkpoint_latest", "model.safetensors.index.json")) or os.path.exists(os.path.join(DRIVE_DIR, "checkpoint_latest", "model.safetensors")):
    load_source = os.path.join(DRIVE_DIR, "checkpoint_latest")
    print(f"📂 Found latest checkpoint in Google Drive: {load_source}")
else:
    load_source = HF_REPO
    print(f"📥 Loading baseline weights from private repo: {load_source}")

model = LlamaForCausalLM.from_pretrained(
    load_source,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="cuda:0"
)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Titan Model Active: {total_params:,} parameters ({total_params / 1e9:.2f} Billion)")
print(f"  • VRAM Footprint: {total_params * 2 / (1024**3):.2f} GB")

## 5. In-Place Adafactor Optimizer & 100-Step Dual Backup Engine

In [ ]:
class InPlaceAdafactor:
    def __init__(self, model, lr=1e-4, eps=(1e-30, 1e-3), clip_threshold=1.0, decay_rate=-0.8, weight_decay=0.01):
        self.model = model
        self.lr = lr
        self.eps = eps
        self.clip_threshold = clip_threshold
        self.decay_rate = decay_rate
        self.weight_decay = weight_decay
        self.step_num = RESUME_STEP
        self.exp_avg_sq = {}
        self.exp_avg_sq_row = {}
        self.exp_avg_sq_col = {}
        self._init_hooks()

    def _init_hooks(self):
        for n, p in self.model.named_parameters():
            if not p.requires_grad:
                continue
            shape = p.data.shape
            dev = p.device
            if len(shape) <= 1:
                self.exp_avg_sq[n] = torch.zeros(shape[0], dtype=torch.float32, device=dev)
            else:
                self.exp_avg_sq_row[n] = torch.zeros(shape[0], dtype=torch.float32, device=dev)
                self.exp_avg_sq_col[n] = torch.zeros(shape[1], dtype=torch.float32, device=dev)

            def make_hook(name, param):
                def hook(p):
                    if p.grad is not None:
                        self._update_param(name, p)
                        p.grad = None  # Immediately reclaim VRAM
                return hook

            if hasattr(p, "register_post_accumulate_grad_hook"):
                p.register_post_accumulate_grad_hook(make_hook(n, p))
            else:
                def make_bwd_hook(name, param):
                    def bwd_hook(grad):
                        param.grad = grad
                        self._update_param(name, param)
                        param.grad = None
                        return None
                    return bwd_hook
                p.register_hook(make_bwd_hook(n, p))

    def _update_param(self, n, p):
        with torch.no_grad():
            grad_fp32 = p.grad.to(torch.float32)
            decay_rate = -self.decay_rate if (self.step_num <= 1 and self.decay_rate < 0) else self.decay_rate
            beta2t = 1.0 - math.pow(max(1, self.step_num), decay_rate)
            update = (grad_fp32 ** 2) + self.eps[0]

            if len(p.data.shape) > 1:
                self.exp_avg_sq_row[n].mul_(beta2t).add_(update.mean(dim=-1), alpha=1.0 - beta2t)
                self.exp_avg_sq_col[n].mul_(beta2t).add_(update.mean(dim=-2), alpha=1.0 - beta2t)
                r_fact = (self.exp_avg_sq_row[n] / self.exp_avg_sq_row[n].mean(dim=-1, keepdim=True)).rsqrt_().unsqueeze(-1)
                c_fact = self.exp_avg_sq_col[n].unsqueeze(-2).rsqrt()
                u = torch.mul(r_fact, c_fact).mul_(grad_fp32)
            else:
                self.exp_avg_sq[n].mul_(beta2t).add_(update, alpha=1.0 - beta2t)
                u = self.exp_avg_sq[n].rsqrt().mul_(grad_fp32)

            u_rms = torch.norm(u, 2.0) / math.sqrt(u.numel())
            u.div_((u_rms / self.clip_threshold).clamp_(min=1.0))

            p_fp32 = p.data.to(torch.float32)
            p_rms = max(self.eps[1], torch.norm(p_fp32, 2.0) / math.sqrt(p.numel()))
            eff_lr = self.lr * p_rms

            if self.weight_decay > 0:
                p_fp32.mul_(1.0 - eff_lr * self.weight_decay)
            p_fp32.add_(u, alpha=-eff_lr)
            p.data.copy_(p_fp32.to(p.dtype))

optimizer = InPlaceAdafactor(model, lr=LEARNING_RATE)
api = HfApi(token=HF_TOKEN)

def backup_checkpoints(step):
    print(f"\n💾 [BACKUP - STEP {step}] Saving 10-step checkpoint to Google Drive & Hugging Face...")
    ckpt_dir = os.path.join(DRIVE_DIR, f"checkpoint_{step}")
    latest_dir = os.path.join(DRIVE_DIR, "checkpoint_latest")
    try:
        os.makedirs(ckpt_dir, exist_ok=True)
        model.save_pretrained(ckpt_dir, max_shard_size="2GB")
        tokenizer.save_pretrained(ckpt_dir)
        os.makedirs(latest_dir, exist_ok=True)
        model.save_pretrained(latest_dir, max_shard_size="2GB")
        tokenizer.save_pretrained(latest_dir)
        print(f"✅ [GOOGLE DRIVE] Checkpoint {step} secured: {ckpt_dir}")
        
        # Keep Drive tidy: retain latest 3 step checkpoints to protect Drive quota
        all_cps = sorted(glob.glob(os.path.join(DRIVE_DIR, "checkpoint_*")), key=lambda x: int(x.split('_')[-1]) if x.split('_')[-1].isdigit() else -1)
        step_cps = [c for c in all_cps if c.split('_')[-1].isdigit()]
        if len(step_cps) > 3:
            for old_c in step_cps[:-3]:
                try:
                    shutil.rmtree(old_c, ignore_errors=True)
                    print(f"🧹 Pruned older Drive checkpoint: {os.path.basename(old_c)}")
                except Exception:
                    pass
    except Exception as e:
        print(f"⚠️ Drive backup note: {e}")

    try:
        print(f"☁️ [HUGGING FACE] Uploading 2GB shards to private repo {HF_REPO}...")
        api.upload_folder(
            folder_path=ckpt_dir,
            repo_id=HF_REPO,
            repo_type="model",
            commit_message=f"Titan Checkpoint Step {step}"
        )
        print(f"✅ [HUGGING FACE] Successfully synced Step {step} to {HF_REPO}")
        # Squash history to keep repo at exactly 1 commit and prevent 50GB storage limit
        try:
            api.super_squash_history(HF_REPO)
            print(f"🧹 [HUGGING FACE] Reclaimed storage quota via super-squash.")
        except Exception:
            pass
    except Exception as e:
        print(f"⚠️ Hugging Face note (Drive backup safe): {e}")
    print("="*70)

## 6. Pre-Training Execution Loop
Prints telemetry every step formatted explicitly as: `completed / total`.

In [ ]:
print(f"🚀 STARTING TITAN TRAINING AT STEP {RESUME_STEP} ON GPU...")
model.train()

global_step = RESUME_STEP
accumulated_loss = 0.0
total_curriculum_steps = total_steps_per_epoch * TARGET_EPOCHS

start_epoch = RESUME_STEP // total_steps_per_epoch
start_micro_step = (RESUME_STEP % total_steps_per_epoch) * GRAD_ACCUM_STEPS

for epoch in range(start_epoch, TARGET_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{TARGET_EPOCHS} ---")
    for micro_step, batch in enumerate(train_loader):
        if epoch == start_epoch and micro_step < start_micro_step:
            continue

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        accumulated_loss += loss.item()

        loss.backward()

        if (micro_step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer.step_num += 1
            global_step += 1

            epoch_completed = global_step % total_steps_per_epoch
            if epoch_completed == 0 and global_step > 0:
                epoch_completed = total_steps_per_epoch
            current_loss = accumulated_loss * GRAD_ACCUM_STEPS
            accumulated_loss = 0.0
            f_b, t_b = torch.cuda.mem_get_info(0)
            u_gb = (t_b - f_b) / (1024**3)
            print(f"Titan Step {global_step} | Epoch {epoch + 1} Progress: {epoch_completed} / {total_steps_per_epoch} | Total Progress: {global_step} / {total_curriculum_steps} | Loss: {current_loss:.4f} | VRAM: {u_gb:.2f} GB / {t_b / (1024**3):.2f} GB ({(u_gb / (t_b / (1024**3))) * 100:.1f}%)")

            # Back up every 10 steps to Google Drive and Hugging Face
            if global_step % BACKUP_EVERY_STEPS == 0:
                backup_checkpoints(global_step)

print("🎉 TITAN 10-EPOCH TRAINING COMPLETE!")
backup_checkpoints(global_step)
print(f"🎓 Final model secured to Google Drive and https://huggingface.co/{HF_REPO}")